# Context-specific control cells

Generate three sets of control cells that share the same GRN but have
different program-activity patterns, so baseline expression reflects
which programs are "on" in each context.

Context overlap design (same structure reused in notebook 09):
- two programs active in **all three** contexts
- three programs **pair-shared** (A+B, B+C, A+C)
- three programs **unique** to one context
- remaining programs inactive in every context

For each context we write:
1. an AnnData of control cells (`./DATA/control_data_<ctx>.h5ad`)
2. a long-format CSV with one row per (program, gene) pair containing
   program activity level and the per-pair W / V values
   (`./DATA/context_<ctx>_programs.csv`)

In [25]:
from libraries import *
from GetGOPrograms import *
from GenerateGRN import *
from ControlDataGenerator import *
from pathlib import Path

## Rebuild `go_gen` and `grn`

`ControlDataGenerator` needs both. Both are built inline here (rather
than loading `grn.pkl`) so this notebook is self-contained and
guaranteed to use 15 programs. GRN parameters mirror notebook 02.

In [26]:
go_gen = GOProgramGenerator(
        n_programs=15,
        organism="human",
        total_genes=5000,
        min_program_size=20,
        max_program_size=150,
        add_synthetic_program_genes=True,
        n_synthetic_per_program=[30, 80],
        synthetic_overlap_mean=0.85,
        synthetic_overlap_std=0.05,
        add_filler_genes=True,
        seed=42,
        data_dir="./go_data",
        use_cached=True,
        verbose=False,
)

programs = go_gen.get_programs(k=15)
all_genes = go_gen.get_all_genes()

grn = GeneRegulatoryNetwork(
    programs=programs,
    all_genes=all_genes,
    w_activation_prob=0.7,
    v_negative_prob=0.7,
    w_mean_range=(0.6, 1.0),
    w_std_range=(0.1, 0.3),
    v_mean_range=(0.15, 0.35),
    v_std_range=(0.05, 0.15),
    min_magnitude=0.01,
    seed=42,
    verbose=False,
)

program_names = list(grn.program_names)
n_programs = len(program_names)
print(f"Programs ({n_programs}):")
for i, p in enumerate(program_names):
    print(f"  [{i:2d}] {p}")

Input sequence provided is already in string format. No operation performed


go_data/go-basic.obo: fmt(1.2) rel(2026-01-23) 42,036 Terms


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequen

Programs (15):
  [ 0] cell_cycle
  [ 1] apoptosis
  [ 2] immune_response
  [ 3] dna_repair
  [ 4] metabolism
  [ 5] ribosome_biogenesis
  [ 6] oxidative_stress
  [ 7] autophagy
  [ 8] cell_adhesion
  [ 9] angiogenesis
  [10] hypoxia_response
  [11] er_stress
  [12] inflammatory_response
  [13] emt
  [14] chromatin_remodeling


In [27]:
program_names

['cell_cycle',
 'apoptosis',
 'immune_response',
 'dna_repair',
 'metabolism',
 'ribosome_biogenesis',
 'oxidative_stress',
 'autophagy',
 'cell_adhesion',
 'angiogenesis',
 'hypoxia_response',
 'er_stress',
 'inflammatory_response',
 'emt',
 'chromatin_remodeling']

## Design per-program activity vectors

`activity_mean` is a length-`n_programs` vector (`ControlDataGenerator`
accepts scalar or vector).  Two independent design dimensions:

1. **Baseline similarity between contexts** is set by `HIGH` and `LOW`
   themselves, because the program contribution to control expression
   goes through `softplus(activity)`:
   `baseline_ratio = softplus(HIGH) / softplus(LOW)`.

2. **Context-specific KO propagation** is set by the simulator's
   sigmoid gate `sigmoid((activity - threshold) · scale)`.  Given a
   target `gate(HIGH) = g_H` and `gate(LOW) = g_L` (gate ratio
   `G = g_H / g_L`), `util.gate_params(HIGH, LOW, g_H, g_L)` solves
   for the matching `(sigmoid_threshold, sigmoid_scale)`.

Defaults below: `HIGH=5, LOW=-1` → baseline ratio ≈ 16; `g_H=0.999,
g_L=0.25` → gate ratio 4 (inactive contexts pass through ~25% of the
KO effect).  Both are stored in `context_design.pkl` for notebook 09.

In [28]:
from util import gate_params

# Baseline-similarity knob (HIGH/LOW; tune these to set softplus ratio)
HIGH, LOW = 5.0, -0.5

# Gate-ratio knob (g_H, g_L; solved into sigmoid_threshold + sigmoid_scale)
GATE_HIGH = 0.999    # gate value at HIGH-activity programs
GATE_LOW  = 0.250    # gate value at LOW-activity programs  -> gate ratio = 4
sigmoid_threshold, sigmoid_scale = gate_params(
    HIGH, LOW, g_H=GATE_HIGH, g_L=GATE_LOW
)

# Report what these choices imply
softplus = lambda a: np.log1p(np.exp(a))
sigmoid  = lambda a: 1.0 / (1.0 + np.exp(-(a - sigmoid_threshold) * sigmoid_scale))
print(f"HIGH={HIGH}, LOW={LOW}")
print(f"  baseline contribution: softplus(HIGH)={softplus(HIGH):.3f}, "
      f"softplus(LOW)={softplus(LOW):.3f}, ratio={softplus(HIGH)/softplus(LOW):.2f}")
print(f"  sigmoid_threshold={sigmoid_threshold:.4f}, sigmoid_scale={sigmoid_scale:.4f}")
print(f"  gate(HIGH)={sigmoid(HIGH):.4f}, gate(LOW)={sigmoid(LOW):.4f}, "
      f"ratio={sigmoid(HIGH)/sigmoid(LOW):.2f}")

def build_activity(active_idx, n_programs):
    a = np.full(n_programs, LOW)
    a[list(set(active_idx))] = HIGH        # dedupe in case of cross-listed indices
    return a

shared_all = [0, 1, 3]
pair_ab    = [4, 5, 11]
pair_bc    = [6, 12]
pair_ac    = [7, 13]
unique_a   = [8, 14]
unique_b   = [9, 2]
unique_c   = [10]

context_active_idx = {
    'A': shared_all + pair_ab + pair_ac + unique_a,
    'B': shared_all + pair_ab + pair_bc + unique_b,
    'C': shared_all + pair_bc + pair_ac + unique_c,
}
activities = {name: build_activity(idx, n_programs)
              for name, idx in context_active_idx.items()}

active_df = pd.DataFrame(
    {name: (activities[name] >= HIGH).astype(int) for name in activities},
    index=program_names,
)

# Order rows: all-three → pair-shared → single → none. Within a tier the row
# pattern (e.g. 110 vs 101) keeps programs with the same context set adjacent.
n_active = active_df.sum(axis=1)
pattern  = active_df.apply(lambda r: tuple(r), axis=1)
order = (pd.DataFrame({'n': n_active, 'p': pattern})
         .sort_values(['n', 'p'], ascending=[False, False])
         .index)
active_df = active_df.loc[order]

cmap = matplotlib.colors.ListedColormap(['lightgrey', 'red'])

fig, ax = plt.subplots(figsize=(8, 0.35 * n_programs + 1))
sns.heatmap(
    active_df,
    cmap=cmap,
    vmin=0,
    vmax=1,
    cbar=False,
    linewidths=0.5,
    linecolor='white',
    ax=ax,
)
ax.set_xlabel('context')
ax.set_ylabel('program')
ax.set_title('Per-context program activity')

legend_handles = [
    matplotlib.patches.Patch(color='red', label='active'),
    matplotlib.patches.Patch(color='lightgrey', label='inactive'),
]
ax.legend(handles=legend_handles, bbox_to_anchor=(1.02, 1), loc='upper left',
          frameon=False)
plt.tight_layout()
plt.show()

ImportError: cannot import name 'gate_params' from 'util' (/home/beraslan/Projects/ModuleFinder/SRC/util.py)

In [ ]:
# Persist artifacts that notebook 09 will load
DATA_DIR = Path('./DATA')
DATA_DIR.mkdir(exist_ok=True)

with open(DATA_DIR / 'grn.pkl', 'wb') as f:
    pickle.dump(grn, f)

with open(DATA_DIR / 'context_design.pkl', 'wb') as f:
    pickle.dump({
        'activities':         activities,          # {ctx: (n_programs,) vec}
        'context_active_idx': context_active_idx,  # {ctx: [program indices]}
        'HIGH':               HIGH,
        'LOW':                LOW,
        'sigmoid_threshold':  sigmoid_threshold,   # solved for target gate ratio
        'sigmoid_scale':      sigmoid_scale,
        'gate_high':          GATE_HIGH,
        'gate_low':           GATE_LOW,
        'program_names':      program_names,
    }, f)

print(f"Saved {DATA_DIR/'grn.pkl'}              ({n_programs} programs)")
print(f"Saved {DATA_DIR/'context_design.pkl'}   (contexts: {list(activities)})")

## Generate control cells per context

Same `ControlDataGenerator` instance (same baseline / dispersion),
different `activity_mean` vector per call.

In [ ]:
DATA_DIR = Path('./DATA')
DATA_DIR.mkdir(exist_ok=True)

ctrl_gen = ControlDataGenerator(grn=grn, go_gen=go_gen, seed=42)

N_CELLS = 1000
context_adatas = {}

for name, activity_vec in activities.items():
    print(f"\n--- context_{name} ---")
    print(f"activity vector: {activity_vec}")
    adata_ctx = ctrl_gen.generate_control_data(
        n_cells=N_CELLS,
        activity_mean=activity_vec,     # <<< per-program mean
        activity_std=0.5,
        use_correlations=True,
        count_distribution='negative_binomial',
        total_counts_per_cell=10000,
    )
    adata_ctx.obs['context'] = f'context_{name}'
    adata_ctx.layers['raw_counts'] = adata_ctx.X.copy()
    adata_ctx.write(DATA_DIR / f'control_data_{name}.h5ad')
    context_adatas[name] = adata_ctx
    print(f"  shape: {adata_ctx.shape}")

## Export combined program-gene table

Single long-format CSV (`./DATA/context_programs.csv`). One row per
(program, gene) pair for every gene that has a non-zero entry in W
for that program. Columns:

1. `program_name`
2. `program_gene`
3. `W_value`              — `grn.W[gene_idx, program_idx]`
4. `V_value`              — `grn.V[program_idx, gene_idx]`
5. `activity_A/B/C`       — scalar activity for this program in each context
6. `active_in_A/B/C`      — boolean flag (`activity > 0`)

A printed summary at the end shows the ON/OFF pattern per program so
you can read off which programs are active in which contexts at a glance.

In [ ]:
records = []
for prog_idx, prog_name in enumerate(program_names):
    gene_indices = np.where(grn.W[:, prog_idx] != 0)[0]
    for gene_idx in gene_indices:
        row = {
            'program_name': prog_name,
            'program_gene':  grn.all_genes[gene_idx],
            'W_value':       float(grn.W[gene_idx, prog_idx]),
            'V_value':       float(grn.V[prog_idx, gene_idx]),
        }
        for ctx_name, activity_vec in activities.items():
            row[f'activity_{ctx_name}'] = float(activity_vec[prog_idx])
            row[f'active_in_{ctx_name}'] = bool(activity_vec[prog_idx] > 0)
        records.append(row)

df = pd.DataFrame(records)
out_path = DATA_DIR / 'context_programs.csv'
df.to_csv(out_path, index=False)
print(f"{out_path}  ({len(df)} rows)")

# Per-program summary: which contexts is each program active in?
program_summary = (df[['program_name'] + [f'active_in_{c}' for c in activities]]
                   .drop_duplicates()
                   .set_index('program_name'))
program_summary.columns = list(activities)
print('\nProgram activity by context:')
print(program_summary.replace({True: 'ON', False: '.'}).to_string())

## Score cells on every program and embed with UMAP

Concatenate the three context AnnData objects, log-normalize for the
gene-set score, then run `sc.tl.score_genes` once per program (gene
list derived from the GRN's `W` matrix).  PCA + neighbors + UMAP, then
plot:

1. UMAP colored by `context` (the three context labels should separate
   or at least cluster distinctly)
2. UMAPs colored by each program score (programs active in a given
   context should light up on the cells from that context)

In [ ]:
# Concatenate all three context controls
adata_all = ad.concat(
    list(context_adatas.values()),
    label='context_origin', keys=list(context_adatas.keys()),
    index_unique='-',
)
adata_all.layers['raw_counts'] = adata_all.X.copy()

# Gene-set scoring needs a log-normalized matrix
sc.pp.normalize_total(adata_all, target_sum=1e4)
sc.pp.log1p(adata_all)

# One score per program
score_cols = []
for prog_idx, prog_name in enumerate(program_names):
    member_idx = np.where(grn.W[:, prog_idx] != 0)[0]
    gene_list = [grn.all_genes[i] for i in member_idx
                 if grn.all_genes[i] in adata_all.var_names]
    if len(gene_list) < 3:
        continue
    score_name = f'score_{prog_name}'
    sc.tl.score_genes(adata_all, gene_list=gene_list,
                      score_name=score_name, random_state=42)
    score_cols.append(score_name)

# PCA -> neighbors -> UMAP on the log-normalized matrix
sc.pp.pca(adata_all, n_comps=min(50, adata_all.shape[1] - 1), random_state=42)
sc.pp.neighbors(adata_all, random_state=42)
sc.tl.umap(adata_all, random_state=42)

# UMAP colored by context
sc.pl.umap(adata_all, color='context', title='UMAP — context label')

# UMAPs colored by per-program score
sc.pl.umap(adata_all, color=score_cols, ncols=4, cmap='viridis', vmin=0, vmax='p99')